# Lesson 3 — Understanding Tokenization

**Understand first. Build second.**

A question-led notebook accompanying the eight-part GitHub lesson. Run cells from top to bottom.


## 1. What is a token?

Our toy tokenizer must preserve words, punctuation, and spaces.


In [ ]:
import re

vocabulary = {"Pisica": 0, "doarme": 1, ".": 2, " ": 3}
reverse_vocabulary = {value: key for key, value in vocabulary.items()}
print(vocabulary)


## 2. Tokenization and vocabulary

The regular expression preserves each whitespace character separately. It is intentionally simplified and not a general Unicode tokenizer.


In [ ]:
def tokenized(text: str) -> list[str]:
    pattern = r"[A-Za-z0-9ăâîșțĂÂÎȘȚ]+|\s|."
    return re.findall(pattern, text)

text = "Pisica   doarme."
tokens = tokenized(text)
print(tokens)
assert tokens == ["Pisica", " ", " ", " ", "doarme", "."]


## 3. Encoding and decoding

An unknown token should raise an error instead of silently disappearing.


In [ ]:
class TokenizerError(Exception):
    pass

class UnknownTokenError(TokenizerError):
    def __init__(self, token: str):
        super().__init__(f"Unknown token: {token!r}")

class UnknownTokenIdError(TokenizerError):
    def __init__(self, token_id: int):
        super().__init__(f"Unknown token ID: {token_id}")

def encode(tokens_list: list[str]) -> list[int]:
    ids = []
    for token in tokens_list:
        if token not in vocabulary:
            raise UnknownTokenError(token)
        ids.append(vocabulary[token])
    return ids

def decode(ids_list: list[int]) -> list[str]:
    tokens = []
    for token_id in ids_list:
        if token_id not in reverse_vocabulary:
            raise UnknownTokenIdError(token_id)
        tokens.append(reverse_vocabulary[token_id])
    return tokens

ids = encode(tokenized(text))
print(ids)
print(decode(ids))
assert ids == [0, 3, 3, 3, 1, 2]


## 4. Can we reconstruct the original text?

**AHA:** The sequence preserves repetition and order. `"".join(...)` reconstructs the original whitespace.


In [ ]:
def run_tokenizer_pipeline(text_input: str) -> str:
    return "".join(decode(encode(tokenized(text_input))))

reconstructed = run_tokenizer_pipeline(text)
print(repr(reconstructed))
assert reconstructed == text
print("Round-trip successful")


## 5. What happens with unknown words?

Our tiny vocabulary cannot encode every word. Subword tokenization is one way real tokenizers reuse smaller text pieces.


In [ ]:
try:
    encode(tokenized("Pisica alearga."))
except UnknownTokenError as error:
    print("Expected error:", error)
else:
    raise AssertionError("An unknown token should have raised an error")


## 6. Same concepts, different languages

The same vocabulary, tokenization, encoding, decoding, and round-trip were implemented in Swift. Compare this notebook with `Tokenization.playground`.


## 7. Exploring real tokenizers

Requires `tiktoken`. Run the install cell if the package is missing. Our original observations were 6 tokens for `cl100k_base` and 5 for `o200k_base` for `"Pisica doarme."`.


In [ ]:
# Run once if needed:
# %pip install tiktoken


In [ ]:
import tiktoken

real_text = "Pisica doarme."
for encoding_name in ["cl100k_base", "o200k_base"]:
    tokenizer = tiktoken.get_encoding(encoding_name)
    real_ids = tokenizer.encode(real_text)
    print(f"\n{encoding_name}: {len(real_ids)} tokens — {real_ids}")
    for token_id in real_ids:
        print(token_id, repr(tokenizer.decode_single_token_bytes(token_id)))
    assert tokenizer.decode(real_ids) == real_text


## 8. Tokens and UTF-8 bytes

A character, a byte, and a token are not the same unit. The `ș` example below demonstrates UTF-8 encoding, **not** a measured `tiktoken` segmentation of Romanian diacritics.


In [ ]:
print("s".encode("utf-8"))
print("ș".encode("utf-8"))

encoded = "ș".encode("utf-8")
first_part, second_part = encoded[:1], encoded[1:]
print(repr(first_part), repr(second_part))

for part in [first_part, second_part]:
    try:
        part.decode("utf-8")
    except UnicodeDecodeError:
        print(repr(part), "is not independently decodable as UTF-8")

assert (first_part + second_part).decode("utf-8") == "ș"
print("Reconstructed:", (first_part + second_part).decode("utf-8"))


## Next question

How do token IDs become **embeddings** that a neural network can learn from?

**Understand first. Build second.**
